In [ ]:
import pandas as pd

drug_nodes = pd.read_pickle("training_data/approved_small_molecule_drugs_review.pkl")
drugs_interactions_train = pd.read_pickle("training_data/drug_protein_interactions_train_review.pkl")
drugs_interactions_validation = pd.read_pickle("training_data/drug_protein_interactions_validation_review.pkl")
drugs_interactions_test = pd.read_pickle("training_data/drug_protein_interactions_test_review.pkl")
drug_indications = pd.read_pickle("training_data/drug_indications_review.pkl")
verified_negatives_protien_interactions_train = pd.read_pickle("training_data/verified_negatives_time_aware_train.pkl")
verified_negatives_protien_interactions_test = pd.read_pickle("training_data/verified_negatives_time_aware_test.pkl")
failed_indication_medium_negatives = pd.read_pickle("training_data/failed_indications_medium.pkl")
failed_indication_hard_negatives = pd.read_pickle("training_data/failed_indications_hard.pkl")
# protien nodes with embeddings
protein_nodes = pd.read_pickle("training_data/protein_nodes_with_embeddings_v4.pkl")

In [2]:
# convert SMILES to molecule object
from rdkit import Chem
import pandas as pd

print("="*80)
print("INVESTIGATING RDKIT MOLECULE OBJECT")
print("="*80)

# Get a sample drug from drug_nodes
sample_drug = drug_nodes.iloc[0]
print(f"\nSample Drug:")
print(f"  Name: {sample_drug['drug_name']}")
print(f"  ChEMBL ID: {sample_drug['drug_id']}")
print(f"  SMILES: {sample_drug['smile']}")

# Convert SMILES to molecule object
smiles = sample_drug['smile']
mol = Chem.MolFromSmiles(smiles)

print(f"\n{'='*80}")
print("MOLECULE OBJECT STRUCTURE")
print(f"{'='*80}")

# Basic info
print(f"\nBasic Properties:")
print(f"  Number of atoms: {mol.GetNumAtoms()}")
print(f"  Number of bonds: {mol.GetNumBonds()}")

# Investigate ATOMS
print(f"\n{'='*80}")
print("ATOMS (First 10)")
print(f"{'='*80}")

for i, atom in enumerate(mol.GetAtoms()):
    if i >= 10:
        print(f"  ... and {mol.GetNumAtoms() - 10} more atoms")
        break
    
    print(f"\nAtom {i}:")
    print(f"  Index:           {atom.GetIdx()}")
    print(f"  Symbol:          {atom.GetSymbol()}")  # C, N, O, etc.
    print(f"  Atomic Number:   {atom.GetAtomicNum()}")  # 6=C, 7=N, 8=O
    print(f"  Degree:          {atom.GetDegree()}")  # Number of bonds
    print(f"  Formal Charge:   {atom.GetFormalCharge()}")
    print(f"  Is Aromatic:     {atom.GetIsAromatic()}")
    print(f"  Hybridization:   {atom.GetHybridization()}")
    print(f"  Total H's:       {atom.GetTotalNumHs()}")
    print(f"  Valence:         {atom.GetTotalValence()}")

# Investigate BONDS
print(f"\n{'='*80}")
print("BONDS (First 10)")
print(f"{'='*80}")

for i, bond in enumerate(mol.GetBonds()):
    if i >= 10:
        print(f"  ... and {mol.GetNumBonds() - 10} more bonds")
        break
    
    atom1 = bond.GetBeginAtom()
    atom2 = bond.GetEndAtom()
    
    print(f"\nBond {i}:")
    print(f"  Connects:      Atom {bond.GetBeginAtomIdx()} ({atom1.GetSymbol()}) -- Atom {bond.GetEndAtomIdx()} ({atom2.GetSymbol()})")
    print(f"  Bond Type:     {bond.GetBondType()}")  # SINGLE, DOUBLE, TRIPLE, AROMATIC
    print(f"  Is Aromatic:   {bond.GetIsAromatic()}")
    print(f"  Is Conjugated: {bond.GetIsConjugated()}")

print("\n" + "="*80)
print("INVESTIGATION COMPLETE!")
print("="*80)
print("\n💡 What we learned:")
print("   - Each ATOM has: symbol (C, N, O...), index, degree, aromaticity")
print("   - Each BOND has: type (SINGLE, DOUBLE, TRIPLE, AROMATIC), connected atoms")
print("\nReady to proceed? Just let me know what you want to keep!")

INVESTIGATING RDKIT MOLECULE OBJECT

Sample Drug:
  Name: CETIRIZINE
  ChEMBL ID: CHEMBL1000
  SMILES: O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1

MOLECULE OBJECT STRUCTURE

Basic Properties:
  Number of atoms: 27
  Number of bonds: 29

ATOMS (First 10)

Atom 0:
  Index:           0
  Symbol:          O
  Atomic Number:   8
  Degree:          1
  Formal Charge:   0
  Is Aromatic:     False
  Hybridization:   SP2
  Total H's:       0
  Valence:         2

Atom 1:
  Index:           1
  Symbol:          C
  Atomic Number:   6
  Degree:          3
  Formal Charge:   0
  Is Aromatic:     False
  Hybridization:   SP2
  Total H's:       0
  Valence:         4

Atom 2:
  Index:           2
  Symbol:          O
  Atomic Number:   8
  Degree:          1
  Formal Charge:   0
  Is Aromatic:     False
  Hybridization:   SP2
  Total H's:       1
  Valence:         2

Atom 3:
  Index:           3
  Symbol:          C
  Atomic Number:   6
  Degree:          2
  Formal Charge:   0
  Is Aromatic:     

In [3]:
from rdkit import Chem
import numpy as np
from tqdm import tqdm

print("="*80)
print("CONVERTING DRUG SMILES TO MOLECULAR GRAPHS")
print("="*80)

def smiles_to_graph(smiles):
    """
    Convert a SMILES string to a molecular graph.
    
    Returns:
        nodes: List of atom dictionaries with features
        edges: List of bond dictionaries (source, target, bond_type)
    """
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        return None, None
    
    # Extract atom nodes
    nodes = []
    for atom in mol.GetAtoms():
        nodes.append({
            "id": atom.GetIdx(),
            "symbol": atom.GetSymbol(),
            "atomic_num": atom.GetAtomicNum(),
            "degree": atom.GetDegree(),
            "formal_charge": atom.GetFormalCharge(),
            "is_aromatic": atom.GetIsAromatic(),
            "hybridization": str(atom.GetHybridization()),
            "num_hs": atom.GetTotalNumHs()
        })
    
    # Extract bond edges (undirected)
    edges = []
    for bond in mol.GetBonds():
        bond_type = str(bond.GetBondType())
        edges.append({
            "source": bond.GetBeginAtomIdx(),
            "target": bond.GetEndAtomIdx(),
            "bond_type": bond_type
        })
    
    return nodes, edges

# Process all drugs
print(f"\nProcessing {len(drug_nodes)} drugs...")

drug_graphs = []
failed_drugs = []

for idx, row in tqdm(drug_nodes.iterrows(), total=len(drug_nodes), desc="Converting SMILES"):
    drug_internal_id = row['drug_internal_id']
    drug_id = row['drug_id']
    drug_name = row.get('drug_name', 'Unknown')
    smiles = row['smile']
    
    # Convert SMILES to graph
    nodes, edges = smiles_to_graph(smiles)
    
    if nodes is None:
        failed_drugs.append({
            'drug_internal_id': drug_internal_id,
            'drug_id': drug_id,
            'drug_name': drug_name,
            'smiles': smiles
        })
        continue
    
    drug_graphs.append({
        'drug_internal_id': drug_internal_id,
        'drug_id': drug_id,
        'drug_name': drug_name,
        'smiles': smiles,
        'num_atoms': len(nodes),
        'num_bonds': len(edges),
        'nodes': nodes,
        'edges': edges
    })

print(f"\n✓ Successfully processed: {len(drug_graphs)} drugs")
print(f"✗ Failed to parse: {len(failed_drugs)} drugs")

# Create summary statistics
print("\n" + "="*80)
print("MOLECULAR GRAPH STATISTICS")
print("="*80)

num_atoms_list = [d['num_atoms'] for d in drug_graphs]
num_bonds_list = [d['num_bonds'] for d in drug_graphs]

print(f"\nNumber of atoms per drug:")
print(f"  Min:    {min(num_atoms_list)}")
print(f"  Max:    {max(num_atoms_list)}")
print(f"  Mean:   {np.mean(num_atoms_list):.1f}")
print(f"  Median: {np.median(num_atoms_list):.1f}")

print(f"\nNumber of bonds per drug:")
print(f"  Min:    {min(num_bonds_list)}")
print(f"  Max:    {max(num_bonds_list)}")
print(f"  Mean:   {np.mean(num_bonds_list):.1f}")
print(f"  Median: {np.median(num_bonds_list):.1f}")

# Bond type distribution
bond_types = []
for d in drug_graphs:
    for edge in d['edges']:
        bond_types.append(edge['bond_type'])

bond_type_counts = pd.Series(bond_types).value_counts()
print(f"\nBond type distribution:")
for bond_type, count in bond_type_counts.items():
    print(f"  {bond_type}: {count:,} ({count/len(bond_types)*100:.1f}%)")

# Atom type distribution
atom_types = []
for d in drug_graphs:
    for node in d['nodes']:
        atom_types.append(node['symbol'])

atom_type_counts = pd.Series(atom_types).value_counts()
print(f"\nTop 10 most common atoms:")
for atom_type, count in atom_type_counts.head(10).items():
    print(f"  {atom_type}: {count:,} ({count/len(atom_types)*100:.1f}%)")

# Save results
print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)


# Example: Show first 3 drug graphs
print("\n" + "="*80)
print("EXAMPLE MOLECULAR GRAPHS (First 3 drugs)")
print("="*80)

for i, drug in enumerate(drug_graphs[:3], 1):
    print(f"\n{i}. {drug['drug_name']} ({drug['drug_id']})")
    print(f"   SMILES: {drug['smiles']}")
    print(f"   Atoms: {drug['num_atoms']}, Bonds: {drug['num_bonds']}")
    
    print(f"\n   Nodes (atoms):")
    for node in drug['nodes'][:5]:  # Show first 5 atoms
        print(f"     {node['id']}: {node['symbol']} (atomic_num={node['atomic_num']}, degree={node['degree']})")
    if len(drug['nodes']) > 5:
        print(f"     ... and {len(drug['nodes']) - 5} more atoms")
    
    print(f"\n   Edges (bonds):")
    for edge in drug['edges'][:5]:  # Show first 5 bonds
        print(f"     {edge['source']} -- {edge['target']} ({edge['bond_type']})")
    if len(drug['edges']) > 5:
        print(f"     ... and {len(drug['edges']) - 5} more bonds")

print("\n" + "="*80)
print("🎉 CONVERSION COMPLETE!")
print("="*80)

CONVERTING DRUG SMILES TO MOLECULAR GRAPHS

Processing 3127 drugs...


Converting SMILES:  63%|██████▎   | 1968/3127 [00:00<00:00, 4451.97it/s][20:53:44] WARNING: not removing hydrogen atom without neighbors
[20:53:44] WARNING: not removing hydrogen atom without neighbors
[20:53:44] WARNING: not removing hydrogen atom without neighbors
[20:53:44] WARNING: not removing hydrogen atom without neighbors
Converting SMILES: 100%|██████████| 3127/3127 [00:00<00:00, 4477.03it/s]


✓ Successfully processed: 3127 drugs
✗ Failed to parse: 0 drugs

MOLECULAR GRAPH STATISTICS

Number of atoms per drug:
  Min:    1
  Max:    200
  Mean:   27.6
  Median: 25.0

Number of bonds per drug:
  Min:    0
  Max:    210
  Mean:   29.0
  Median: 27.0

Bond type distribution:
  SINGLE: 53,356 (58.9%)
  AROMATIC: 30,161 (33.3%)
  DOUBLE: 6,908 (7.6%)
  TRIPLE: 152 (0.2%)

Top 10 most common atoms:
  C: 60,903 (70.6%)
  O: 13,321 (15.5%)
  N: 7,879 (9.1%)
  Cl: 1,106 (1.3%)
  S: 1,040 (1.2%)
  F: 1,028 (1.2%)
  Na: 273 (0.3%)
  I: 228 (0.3%)
  P: 158 (0.2%)
  Br: 101 (0.1%)

SAVING RESULTS

EXAMPLE MOLECULAR GRAPHS (First 3 drugs)

1. CETIRIZINE (CHEMBL1000)
   SMILES: O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1
   Atoms: 27, Bonds: 29

   Nodes (atoms):
     0: O (atomic_num=8, degree=1)
     1: C (atomic_num=6, degree=3)
     2: O (atomic_num=8, degree=1)
     3: C (atomic_num=6, degree=2)
     4: O (atomic_num=8, degree=2)
     ... and 22 more atoms

   Edges (bonds):
     0 --

In [4]:
from collections import Counter
import pandas as pd

print("="*80)
print("COMPREHENSIVE ANALYSIS: ALL ATOM & BOND FEATURES")
print("="*80)

# Collect ALL features from atoms and bonds
atom_symbols = []
atom_degrees = []
atom_formal_charges = []
atom_is_aromatic = []
atom_hybridizations = []
atom_num_hs = []

bond_types = []

for drug in drug_graphs:
    # Collect ALL atom features
    for node in drug['nodes']:
        atom_symbols.append(node['symbol'])
        atom_degrees.append(node['degree'])
        atom_formal_charges.append(node['formal_charge'])
        atom_is_aromatic.append(node['is_aromatic'])
        atom_hybridizations.append(node['hybridization'])
        atom_num_hs.append(node['num_hs'])
    
    # Collect bond types
    for edge in drug['edges']:
        bond_types.append(edge['bond_type'])

# Count unique values
print("\n" + "="*80)
print("1️⃣  ATOM FEATURES")
print("="*80)

print(f"\n📊 Symbol (atom type):")
symbol_counts = Counter(atom_symbols)
print(f"   Total unique: {len(symbol_counts)}")
print(f"   Values: {sorted(symbol_counts.keys())}")
for symbol, count in sorted(symbol_counts.items(), key=lambda x: -x[1]):
    print(f"      {symbol:3s}: {count:6,} atoms ({count/len(atom_symbols)*100:.1f}%)")

print(f"\n📊 Degree (number of bonds):")
degree_counts = Counter(atom_degrees)
print(f"   Total unique: {len(degree_counts)}")
print(f"   Values: {sorted(degree_counts.keys())}")
for deg, count in sorted(degree_counts.items()):
    print(f"      {deg}: {count:,} atoms ({count/len(atom_degrees)*100:.1f}%)")

print(f"\n📊 Formal Charge:")
charge_counts = Counter(atom_formal_charges)
print(f"   Total unique: {len(charge_counts)}")
print(f"   Values: {sorted(charge_counts.keys())}")
for charge, count in sorted(charge_counts.items()):
    print(f"      {charge:+d}: {count:,} atoms ({count/len(atom_formal_charges)*100:.1f}%)")

print(f"\n📊 Is Aromatic:")
aromatic_counts = Counter(atom_is_aromatic)
print(f"   Total unique: {len(aromatic_counts)}")
for val, count in aromatic_counts.items():
    print(f"      {val}: {count:,} atoms ({count/len(atom_is_aromatic)*100:.1f}%)")

print(f"\n📊 Hybridization:")
hybrid_counts = Counter(atom_hybridizations)
print(f"   Total unique: {len(hybrid_counts)}")
print(f"   Values: {sorted(hybrid_counts.keys())}")
for hybrid, count in sorted(hybrid_counts.items(), key=lambda x: -x[1]):
    print(f"      {hybrid:10s}: {count:,} atoms ({count/len(atom_hybridizations)*100:.1f}%)")

print(f"\n📊 Number of Hydrogens:")
hs_counts = Counter(atom_num_hs)
print(f"   Total unique: {len(hs_counts)}")
print(f"   Values: {sorted(hs_counts.keys())}")
for hs, count in sorted(hs_counts.items()):
    print(f"      {hs}: {count:,} atoms ({count/len(atom_num_hs)*100:.1f}%)")

print("\n" + "="*80)
print("2️⃣  BOND FEATURES")
print("="*80)

print(f"\n📊 Bond Type:")
bond_counts = Counter(bond_types)
print(f"   Total unique: {len(bond_counts)}")
print(f"   Values: {sorted(bond_counts.keys())}")
for bond, count in sorted(bond_counts.items(), key=lambda x: -x[1]):
    print(f"      {bond:10s}: {count:,} bonds ({count/len(bond_types)*100:.1f}%)")

print("\n" + "="*80)
print("📋 SUMMARY FOR ONE-HOT ENCODING")
print("="*80)

print(f"\n✅ ATOM FEATURES TO ENCODE:")
print(f"   1. Symbol:         {len(symbol_counts):2d} unique values -> {len(symbol_counts):3d} dimensions")
print(f"   2. Degree:         {len(degree_counts):2d} unique values -> {len(degree_counts):3d} dimensions")
print(f"   3. Formal Charge:  {len(charge_counts):2d} unique values -> {len(charge_counts):3d} dimensions")
print(f"   4. Is Aromatic:    {len(aromatic_counts):2d} unique values -> {len(aromatic_counts):3d} dimensions")
print(f"   5. Hybridization:  {len(hybrid_counts):2d} unique values -> {len(hybrid_counts):3d} dimensions")
print(f"   6. Num Hydrogens:  {len(hs_counts):2d} unique values -> {len(hs_counts):3d} dimensions")

total_atom_features = (len(symbol_counts) + len(degree_counts) + 
                       len(charge_counts) + len(aromatic_counts) + 
                       len(hybrid_counts) + len(hs_counts))

print(f"\n   📊 Total ATOM embedding size: {total_atom_features} dimensions (one-hot)")

print(f"\n✅ BOND FEATURES TO ENCODE:")
print(f"   1. Bond Type:      {len(bond_counts):2d} unique values -> {len(bond_counts):3d} dimensions")

print(f"\n   📊 Total BOND embedding size: {len(bond_counts)} dimensions (one-hot)")

print("\n" + "="*80)
print("🎯 EMBEDDING BREAKDOWN")
print("="*80)
print(f"\nEach ATOM will be represented as a {total_atom_features}-dimensional vector")
print(f"Each BOND will be represented as a {len(bond_counts)}-dimensional vector")

print(f"\nExample for one drug molecule:")
sample_drug = drug_graphs[0]
print(f"  Drug: {sample_drug['drug_name']}")
print(f"  Atoms: {sample_drug['num_atoms']} x {total_atom_features} dims = {sample_drug['num_atoms'] * total_atom_features:,} values")
print(f"  Bonds: {sample_drug['num_bonds']} x {len(bond_counts)} dims = {sample_drug['num_bonds'] * len(bond_counts):,} values")

print("\n" + "="*80)
print("🎯 READY FOR ONE-HOT ENCODING!")
print("="*80)

COMPREHENSIVE ANALYSIS: ALL ATOM & BOND FEATURES

1️⃣  ATOM FEATURES

📊 Symbol (atom type):
   Total unique: 32
   Values: ['Ag', 'Al', 'As', 'B', 'Ba', 'Bi', 'Br', 'C', 'Ca', 'Cl', 'F', 'Ga', 'H', 'He', 'I', 'K', 'Kr', 'Li', 'Mg', 'N', 'Na', 'O', 'P', 'Ra', 'Rb', 'S', 'Se', 'Si', 'Sr', 'Xe', 'Yb', 'Zn']
      C  : 60,903 atoms (70.6%)
      O  : 13,321 atoms (15.5%)
      N  :  7,879 atoms (9.1%)
      Cl :  1,106 atoms (1.3%)
      S  :  1,040 atoms (1.2%)
      F  :  1,028 atoms (1.2%)
      Na :    273 atoms (0.3%)
      I  :    228 atoms (0.3%)
      P  :    158 atoms (0.2%)
      Br :    101 atoms (0.1%)
      K  :     35 atoms (0.0%)
      H  :     30 atoms (0.0%)
      Ca :     25 atoms (0.0%)
      Mg :     22 atoms (0.0%)
      Si :     14 atoms (0.0%)
      Li :      8 atoms (0.0%)
      Zn :      7 atoms (0.0%)
      Sr :      5 atoms (0.0%)
      B  :      5 atoms (0.0%)
      Se :      4 atoms (0.0%)
      Xe :      4 atoms (0.0%)
      As :      3 atoms (0.0%)
      Al :

In [5]:
# Create feature mappings for one-hot encoding
atom_feature_mappings = {
    'symbol': {val: i for i, val in enumerate(sorted(symbol_counts.keys()))},
    'degree': {val: i for i, val in enumerate(sorted(degree_counts.keys()))},
    'formal_charge': {val: i for i, val in enumerate(sorted(charge_counts.keys()))},
    'is_aromatic': {val: i for i, val in enumerate(sorted(aromatic_counts.keys()))},
    'hybridization': {val: i for i, val in enumerate(sorted(hybrid_counts.keys()))},
    'num_hs': {val: i for i, val in enumerate(sorted(hs_counts.keys()))}
}

bond_feature_mappings = {
    'bond_type': {val: i for i, val in enumerate(sorted(bond_counts.keys()))}
}

# Calculate total dimensions
total_atom_dim = sum(len(mapping) for mapping in atom_feature_mappings.values())
total_bond_dim = len(bond_feature_mappings['bond_type'])

print(f"\n✅ Feature mappings created:")
print(f"   Total atom dimensions: {total_atom_dim}")
print(f"   Total bond dimensions: {total_bond_dim}")

def encode_atom(atom_node):
    """Convert atom features to one-hot encoded vector"""
    vector = np.zeros(total_atom_dim)
    offset = 0
    for feature_name, mapping in atom_feature_mappings.items():
        value = atom_node[feature_name]
        if value in mapping:
            idx = offset + mapping[value]
            vector[idx] = 1.0
        offset += len(mapping)
    return vector

def encode_bond(bond_edge):
    """Convert bond features to one-hot encoded vector"""
    vector = np.zeros(total_bond_dim)
    bond_type = bond_edge['bond_type']
    if bond_type in bond_feature_mappings['bond_type']:
        idx = bond_feature_mappings['bond_type'][bond_type]
        vector[idx] = 1.0
    return vector

# Test encoding
sample_drug = drug_graphs[0]
first_atom = sample_drug['nodes'][0]
first_bond = sample_drug['edges'][0]

encoded_atom = encode_atom(first_atom)
encoded_bond = encode_bond(first_bond)

print(f"\n✅ Test encoding successful!")
print(f"   Atom vector shape: {encoded_atom.shape}")
print(f"   Bond vector shape: {encoded_bond.shape}")

# Encode all drugs
for drug in drug_graphs:
    # Node features matrix
    if drug['nodes']:
        node_attr = np.vstack([encode_atom(n) for n in drug['nodes']])
    else:
        node_attr = np.zeros((0, total_atom_dim), dtype=float)

    # Bond features and connectivity
    if drug['edges']:
        bond_features = np.vstack([encode_bond(e) for e in drug['edges']])
        sources = [int(e['source']) for e in drug['edges']]
        targets = [int(e['target']) for e in drug['edges']]

        # Bidirectional edges
        edge_index = np.array([sources + targets, targets + sources], dtype=np.int64)
        edge_attr = np.vstack([bond_features, bond_features])
    else:
        edge_index = np.zeros((2, 0), dtype=np.int64)
        edge_attr = np.zeros((0, total_bond_dim), dtype=float)

    # Save to drug dict
    drug['node_attr'] = node_attr
    drug['edge_attr'] = edge_attr
    drug['edge_index'] = edge_index

# Example output
sample = drug_graphs[0]
print("\n" + "="*80)
print(f"EXAMPLE: {sample['drug_name']}")
print("="*80)
print(f"  node_attr shape:  {sample['node_attr'].shape}")
print(f"  edge_attr shape:  {sample['edge_attr'].shape}")
print(f"  edge_index shape: {sample['edge_index'].shape}")
print("="*80)


✅ Feature mappings created:
   Total atom dimensions: 57
   Total bond dimensions: 4

✅ Test encoding successful!
   Atom vector shape: (57,)
   Bond vector shape: (4,)

EXAMPLE: CETIRIZINE
  node_attr shape:  (27, 57)
  edge_attr shape:  (58, 4)
  edge_index shape: (2, 58)


In [6]:
# Encode all drugs
for drug in drug_graphs:
    drug['encoded_atoms'] = [encode_atom(node) for node in drug['nodes']]
    drug['encoded_bonds'] = [encode_bond(edge) for edge in drug['edges']]

In [7]:
drug_graph_embeddings = []

for drug in drug_graphs:
    clean_drug = {
        # Identifiers (for tracking)
        'drug_internal_id': drug['drug_internal_id'],
        'drug_id': drug['drug_id'],
        'drug_name': drug['drug_name'],
        
        # Graph structure - ALL the model needs
        'node_attr': drug['node_attr'],      # (num_atoms, 57)
        'edge_attr': drug['edge_attr'],      # (num_edges*2, 4)
        'edge_index': drug['edge_index']     # (2, num_edges*2)
    }
    
    drug_graph_embeddings.append(clean_drug)

print(f"\n✓ Created {len(drug_graph_embeddings)} minimal drug embeddings")

# Show example
drug_graph_embeddings


✓ Created 3127 minimal drug embeddings


[{'drug_internal_id': 111185,
  'drug_id': 'CHEMBL1000',
  'drug_name': 'CETIRIZINE',
  'node_attr': array([[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 1., 0., 0.],
         ...,
         [0., 0., 0., ..., 1., 0., 0.],
         [0., 0., 0., ..., 0., 1., 0.],
         [0., 0., 0., ..., 0., 1., 0.]]),
  'edge_attr': array([[0., 1., 0., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [1., 0., 0., 0.],
         [1., 0., 0., 0.],
         [1., 0., 0., 0.],
         [1., 0., 0., 0.],
         [1., 0., 0., 0.],
         [0., 0., 1., 0.],
         [1., 0., 0., 0.],
         [1., 0., 0., 0.],
         [1., 0., 0., 0.],
         [0., 0., 1., 0.],
         [1., 0., 0., 0.],
         [1

In [8]:
import torch
from torch_geometric.data import Data
from tqdm import tqdm

print("="*80)
print("CONVERTING DRUGS TO PYTORCH GEOMETRIC DATA OBJECTS")
print("="*80)

# Convert each drug to a PyG Data object
drug_pyg_objects = []

# Find the cell where you create drug_pyg_objects (around line 1730)
# REPLACE IT WITH THIS:

drug_pyg_objects = []
failed_drugs = []

for drug in tqdm(drug_graph_embeddings, desc="Creating PyG objects"):
    # Validate BEFORE converting to tensors
    node_attr = drug['node_attr']
    edge_attr = drug['edge_attr']
    edge_index = drug['edge_index']
    
    # Check for NaN/Inf
    if np.isnan(node_attr).any() or np.isinf(node_attr).any():
        failed_drugs.append((drug['drug_id'], 'NaN/Inf in node_attr'))
        continue
    
    if np.isnan(edge_attr).any() or np.isinf(edge_attr).any():
        failed_drugs.append((drug['drug_id'], 'NaN/Inf in edge_attr'))
        continue
    
    # Check edge indices are valid
    num_atoms = node_attr.shape[0]
    if (edge_index >= num_atoms).any() or (edge_index < 0).any():
        failed_drugs.append((drug['drug_id'], f'Invalid edge_index (max={edge_index.max()}, num_atoms={num_atoms})'))
        continue
    
    # Create PyG Data object (ON CPU)
    x = torch.FloatTensor(node_attr)
    edge_index = torch.LongTensor(edge_index)
    edge_attr = torch.FloatTensor(edge_attr)
    
    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        drug_internal_id=drug['drug_internal_id'],
        drug_id=drug['drug_id'],
        drug_name=drug['drug_name']
    )
    
    drug_pyg_objects.append(data)

print(f"\n✓ Created {len(drug_pyg_objects)} valid PyG objects")
print(f"✗ Failed: {len(failed_drugs)} drugs")

if failed_drugs:
    print("\nFailed drugs:")
    for drug_id, reason in failed_drugs[:10]:
        print(f"  {drug_id}: {reason}")

# Show example
print("\n" + "="*80)
print(f"EXAMPLE: {drug_pyg_objects[0].drug_name}")
print("="*80)
print(f"  drug_internal_id: {drug_pyg_objects[0].drug_internal_id}")
print(f"  drug_id:          {drug_pyg_objects[0].drug_id}")
print(f"  drug_name:        {drug_pyg_objects[0].drug_name}")
print(f"  x (node features): {drug_pyg_objects[0].x.shape}")
print(f"  edge_index:        {drug_pyg_objects[0].edge_index.shape}")
print(f"  edge_attr:         {drug_pyg_objects[0].edge_attr.shape}")
print(f"  num_nodes:         {drug_pyg_objects[0].num_nodes}")
print(f"  num_edges:         {drug_pyg_objects[0].num_edges}")

print("\n💾 Sample node features (first atom):")
print(drug_pyg_objects[0].x[0])  # one-hot encoded features

print("\n💾 Sample edge_index (first 5 edges):")
print(drug_pyg_objects[0].edge_index[:, :5])

print("\n" + "="*80)
print("✅ READY FOR GNN MODELING!")
print("="*80)
print("\nEach drug is now a PyG Data object with:")
print("  • x:          node features (atom embeddings)")
print("  • edge_index: graph connectivity")
print("  • edge_attr:  edge features (bond types)")
print("  • metadata:   drug_internal_id, drug_id, drug_name")
print("\n🎯 Next: Build GNN model to encode these into embeddings!")

/home/joe/projects/pharmacology-graph/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CONVERTING DRUGS TO PYTORCH GEOMETRIC DATA OBJECTS


Creating PyG objects: 100%|██████████| 3127/3127 [00:00<00:00, 27120.85it/s]


✓ Created 3127 valid PyG objects
✗ Failed: 0 drugs

EXAMPLE: CETIRIZINE
  drug_internal_id: 111185
  drug_id:          CHEMBL1000
  drug_name:        CETIRIZINE
  x (node features): torch.Size([27, 57])
  edge_index:        torch.Size([2, 58])
  edge_attr:         torch.Size([58, 4])
  num_nodes:         27
  num_edges:         58

💾 Sample node features (first atom):
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
        0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 1.,
        0., 0., 0.])

💾 Sample edge_index (first 5 edges):
tensor([[0, 1, 1, 3, 4],
        [1, 2, 3, 4, 5]])

✅ READY FOR GNN MODELING!

Each drug is now a PyG Data object with:
  • x:          node features (atom embeddings)
  • edge_index: graph connectivity
  • edge_attr:  edge features (bond types)
  • metadata:   drug_internal_id, drug_id, drug_name

🎯 Next: Build GNN model to en

In [9]:

print("="*80)
print("CONVERTING PROTEINS TO PYTORCH GEOMETRIC DATA OBJECTS")
print("="*80)

# Convert each protein to a PyG Data object
protein_pyg_objects = []

for idx, row in tqdm(protein_nodes.iterrows(), total=len(protein_nodes), desc="Creating PyG objects"):
    embedding_str = row['esm2_embedding']
    embedding = np.array(embedding_str)
    x = torch.FloatTensor(embedding).unsqueeze(0)  # Shape: (1, 1280)
    data = Data(
        x=x,  # node features (just one node per protein)
        protein_internal_id=int(row['protein_internal_id']),
        protein_id=str(row['protein_id']),
        protein_name=str(row['protein_name']),
        uniprot_id=str(row['uniprot_id']),
        sequence_length=int(row['sequence_length'])
    )
    
    protein_pyg_objects.append(data)

print(f"\n✓ Created {len(protein_pyg_objects)} PyTorch Geometric Data objects")

# Show example
print("\n" + "="*80)
print(f"EXAMPLE: {protein_pyg_objects[0].protein_name}")
print("="*80)
print(f"  protein_internal_id: {protein_pyg_objects[0].protein_internal_id}")
print(f"  protein_id:          {protein_pyg_objects[0].protein_id}")
print(f"  protein_name:        {protein_pyg_objects[0].protein_name}")
print(f"  uniprot_id:          {protein_pyg_objects[0].uniprot_id}")
print(f"  sequence_length:     {protein_pyg_objects[0].sequence_length}")
print(f"  x (embedding):       {protein_pyg_objects[0].x.shape}")

print("\n💾 Sample embedding (first 10 values):")
print(protein_pyg_objects[0].x[0, :10])



CONVERTING PROTEINS TO PYTORCH GEOMETRIC DATA OBJECTS


Creating PyG objects: 100%|██████████| 1156/1156 [00:00<00:00, 5368.23it/s]


✓ Created 1156 PyTorch Geometric Data objects

EXAMPLE: Maltase-glucoamylase
  protein_internal_id: 1
  protein_id:          CHEMBL2074
  protein_name:        Maltase-glucoamylase
  uniprot_id:          O43451
  sequence_length:     2753
  x (embedding):       torch.Size([1, 2560])

💾 Sample embedding (first 10 values):
tensor([ 0.0474,  0.0507, -0.0052, -0.0766, -0.0222, -0.0937, -0.0363,  0.0933,
         0.0464, -0.0282])


In [10]:
drug_effects = drug_indications

In [11]:
# Get unique effects from drug_effects
unique_effects = drug_effects[['effect_id', 'effect_name']].drop_duplicates()
print(f"\nTotal unique effects: {len(unique_effects)}")


Total unique effects: 1065


In [12]:

# Create simple embeddings for effects (we'll use one-hot encoding or learned embeddings later)
# For now, create random embeddings as placeholders
EFFECT_EMBEDDING_DIM = 32 # change if number of effects is bigger

effect_pyg_objects = []

for idx, row in tqdm(unique_effects.iterrows(), total=len(unique_effects), desc="Creating effect PyG objects"):
    effect_embedding = torch.randn(1, EFFECT_EMBEDDING_DIM)
    
    data = Data(
        x=effect_embedding,  # (1, 32)
        effect_id=str(row['effect_id']),
        effect_name=str(row['effect_name'])
    )
    
    effect_pyg_objects.append(data)

print(f"\n✓ Created {len(effect_pyg_objects)} effect PyG Data objects")

# Show example
print("\n" + "="*80)
print(f"EXAMPLE: {effect_pyg_objects[0].effect_name}")
print("="*80)
print(f"  effect_id:      {effect_pyg_objects[0].effect_id}")
print(f"  effect_name:    {effect_pyg_objects[0].effect_name}")
print(f"  x (embedding):  {effect_pyg_objects[0].x.shape}")

print("\n💾 Sample embedding (first 10 values):")
print(effect_pyg_objects[0].x[0, :10])

print("\n" + "="*80)
print("CREATING DRUG-EFFECT EDGE INDEX")
print("="*80)

# Create mappings: drug_internal_id -> index in drug_pyg_objects
drug_internal_id_to_idx = {}
for i, drug_data in enumerate(drug_pyg_objects):
    drug_internal_id_to_idx[drug_data.drug_internal_id] = i

# Create mappings: effect_id -> index in effect_pyg_objects
effect_id_to_idx = {}
for i, effect_data in enumerate(effect_pyg_objects):
    effect_id_to_idx[effect_data.effect_id] = i

print(f"\nDrug mapping: {len(drug_internal_id_to_idx)} drugs")
print(f"Effect mapping: {len(effect_id_to_idx)} effects")

# Create edge list: drug -> effect
drug_effect_edges = []
edge_attributes = []

for _, row in tqdm(drug_effects.iterrows(), total=len(drug_effects), desc="Creating drug-effect edges"):
    drug_internal_id = int(row['drug_internal_id'])
    effect_id = str(row['effect_id'])
    
    # Check if both exist in our mappings
    if drug_internal_id in drug_internal_id_to_idx and effect_id in effect_id_to_idx:
        drug_idx = drug_internal_id_to_idx[drug_internal_id]
        effect_idx = effect_id_to_idx[effect_id]
        
        # Add edge: drug -> effect
        drug_effect_edges.append([drug_idx, effect_idx])
        
        # Store edge attributes (indication_phase)
        edge_attributes.append({
            'indication_phase': float(row.get('indication_phase', 4.0)),
            'num_references': int(row.get('num_references', 1))
        })

# Convert to tensors
drug_effect_edge_index = torch.LongTensor(drug_effect_edges).t().contiguous()  # Shape: (2, num_edges)

print(f"\n✓ Created {drug_effect_edge_index.shape[1]} drug-effect edges")
print(f"  Edge index shape: {drug_effect_edge_index.shape}")

# Create edge attribute tensor
indication_phases = torch.FloatTensor([attr['indication_phase'] for attr in edge_attributes])
num_references = torch.FloatTensor([attr['num_references'] for attr in edge_attributes])

drug_effect_edge_attr = torch.stack([indication_phases, num_references], dim=1)  # Shape: (num_edges, 2)
print(f"  Edge attr shape: {drug_effect_edge_attr.shape}")

print("\n💾 Sample edges (first 5):")
print(f"  Edge index:\n{drug_effect_edge_index[:, :5]}")
print(f"  Edge attributes:\n{drug_effect_edge_attr[:5]}")

print("\n" + "="*80)
print("✅ DRUG-EFFECT GRAPH READY!")
print("="*80)
print("\nSummary:")
print(f"  • {len(drug_pyg_objects)} drug nodes (with molecular graphs)")
print(f"  • {len(effect_pyg_objects)} effect nodes (with embeddings)")
print(f"  • {drug_effect_edge_index.shape[1]} drug→effect edges")



Creating effect PyG objects: 100%|██████████| 1065/1065 [00:00<00:00, 44174.15it/s]



✓ Created 1065 effect PyG Data objects

EXAMPLE: Eye Manifestations
  effect_id:      D005132
  effect_name:    Eye Manifestations
  x (embedding):  torch.Size([1, 32])

💾 Sample embedding (first 10 values):
tensor([-0.5304,  0.3287,  0.2188,  0.4033, -1.2342,  0.7724,  0.5031,  1.0222,
        -0.4111, -0.8037])

CREATING DRUG-EFFECT EDGE INDEX

Drug mapping: 3127 drugs
Effect mapping: 1065 effects


Creating drug-effect edges: 100%|██████████| 7086/7086 [00:00<00:00, 82071.61it/s]


✓ Created 5633 drug-effect edges
  Edge index shape: torch.Size([2, 5633])
  Edge attr shape: torch.Size([5633, 2])

💾 Sample edges (first 5):
  Edge index:
tensor([[0, 0, 1, 3, 3],
        [0, 1, 2, 3, 4]])
  Edge attributes:
tensor([[4., 1.],
        [4., 1.],
        [4., 1.],
        [4., 1.],
        [4., 1.]])

✅ DRUG-EFFECT GRAPH READY!

Summary:
  • 3127 drug nodes (with molecular graphs)
  • 1065 effect nodes (with embeddings)
  • 5633 drug→effect edges


In [13]:
print("="*80)
print("CREATING COMPREHENSIVE NODE & EDGE MAPPINGS")
print("="*80)

# Save all mappings for future use
graph_data = {
    # Nodes
    'drug_pyg_objects': drug_pyg_objects,
    'effect_pyg_objects': effect_pyg_objects,
    'protein_pyg_objects': protein_pyg_objects,  # Already created
    
    # Node mappings
    'drug_internal_id_to_idx': drug_internal_id_to_idx,
    'effect_id_to_idx': effect_id_to_idx,
    
    # Edges: Drug → Effect
    'drug_effect_edge_index': drug_effect_edge_index,
    'drug_effect_edge_attr': drug_effect_edge_attr,
    
    # Metadata
    'num_drugs': len(drug_pyg_objects),
    'num_effects': len(effect_pyg_objects),
    'num_proteins': len(protein_pyg_objects),
    'num_drug_effect_edges': drug_effect_edge_index.shape[1]
}

print("\n✅ Graph data structure created:")
print(f"  Drugs:           {graph_data['num_drugs']:,}")
print(f"  Effects:         {graph_data['num_effects']:,}")
print(f"  Proteins:        {graph_data['num_proteins']:,}")
print(f"  Drug→Effect:     {graph_data['num_drug_effect_edges']:,}")



CREATING COMPREHENSIVE NODE & EDGE MAPPINGS

✅ Graph data structure created:
  Drugs:           3,127
  Effects:         1,065
  Proteins:        1,156
  Drug→Effect:     5,633


In [14]:
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import HeteroData

print("="*80)
print("STEP 1: BUILD NODE MAPPINGS")
print("="*80)

drug_ids = drug_nodes['drug_internal_id'].values
protein_ids = protein_nodes['protein_id'].unique()
effect_ids = drug_indications['effect_id'].unique()

drug_to_idx = {did: i for i, did in enumerate(drug_ids)}
protein_to_idx = {pid: i for i, pid in enumerate(protein_ids)}
effect_to_idx = {eid: i for i, eid in enumerate(effect_ids)}

num_drugs = len(drug_to_idx)
num_proteins = len(protein_to_idx)
num_effects = len(effect_to_idx)

print(f"  Drugs:    {num_drugs:,}")
print(f"  Proteins: {num_proteins:,}")
print(f"  Effects:  {num_effects:,}")

STEP 1: BUILD NODE MAPPINGS
  Drugs:    3,127
  Proteins: 1,156
  Effects:  1,065


In [15]:
print("\n" + "="*80)
print("STEP 1.5: FILTER TO CONNECTED NODES ONLY")
print("="*80)

# ── Collect all nodes that appear in at least ONE edge across ALL splits ──
all_dp_nodes_src = set()
all_dp_nodes_tgt = set()
all_di_nodes_src = set()
all_di_nodes_tgt = set()

for _, row in drugs_interactions_train.iterrows():
    all_dp_nodes_src.add(int(row['drug_internal_id']))
    all_dp_nodes_tgt.add(str(row['protein_id']))

for _, row in drugs_interactions_validation.iterrows():
    all_dp_nodes_src.add(int(row['drug_internal_id']))
    all_dp_nodes_tgt.add(str(row['protein_id']))

for _, row in drugs_interactions_test.iterrows():
    all_dp_nodes_src.add(int(row['drug_internal_id']))
    all_dp_nodes_tgt.add(str(row['protein_id']))

for _, row in drug_indications.iterrows():
    all_di_nodes_src.add(int(row['drug_internal_id']))
    all_di_nodes_tgt.add(str(row['effect_id']))

# Union of all connected nodes (no feature-availability filtering)
connected_drugs = all_dp_nodes_src | all_di_nodes_src
connected_proteins = all_dp_nodes_tgt
connected_effects = all_di_nodes_tgt

print(f"  Connected nodes found in edges:")
print(f"    Drugs:    {len(connected_drugs):,}")
print(f"    Proteins: {len(connected_proteins):,}")
print(f"    Effects:  {len(connected_effects):,}")

# ── Filter and reindex nodes ──
drug_ids_filtered = np.array(sorted(list(connected_drugs)))
protein_ids_filtered = np.array(sorted(list(connected_proteins)))
effect_ids_filtered = np.array(sorted(list(connected_effects)))

# Create NEW mappings with only connected nodes
drug_to_idx = {did: i for i, did in enumerate(drug_ids_filtered)}
protein_to_idx = {pid: i for i, pid in enumerate(protein_ids_filtered)}
effect_to_idx = {eid: i for i, eid in enumerate(effect_ids_filtered)}

num_drugs = len(drug_to_idx)
num_proteins = len(protein_to_idx)
num_effects = len(effect_to_idx)

total_nodes = num_drugs + num_proteins + num_effects
print(f"\n  Final node counts:")
print(f"    Drugs:    {num_drugs:,}")
print(f"    Proteins: {num_proteins:,}")
print(f"    Effects:  {num_effects:,}")
print(f"    Total:    {total_nodes:,}")

print("\n" + "="*80)
print("✓ FILTERED TO CONNECTED NODES ONLY (identity features — no SMILES/ESM2 needed)")
print("="*80)


STEP 1.5: FILTER TO CONNECTED NODES ONLY
  Connected nodes found in edges:
    Drugs:    3,071
    Proteins: 1,966
    Effects:  1,065

  Final node counts:
    Drugs:    3,071
    Proteins: 1,966
    Effects:  1,065
    Total:    6,102

✓ FILTERED TO CONNECTED NODES ONLY (identity features — no SMILES/ESM2 needed)


In [16]:
print("\n" + "="*80)
print("STEP 2: PROCESS DRUG-PROTEIN EDGES (TIME-AWARE SPLIT)")
print("="*80)

def edges_to_tensor(df, src_col, tgt_col, src_map, tgt_map):
    """Convert edges to tensor, filtering out missing nodes."""
    src_indices = []
    tgt_indices = []
    for _, row in df.iterrows():
        src_id = int(row[src_col])
        tgt_id = str(row[tgt_col])
        if src_id in src_map and tgt_id in tgt_map:
            src_indices.append(src_map[src_id])
            tgt_indices.append(tgt_map[tgt_id])
    return torch.LongTensor([src_indices, tgt_indices]) if src_indices else torch.zeros((2, 0), dtype=torch.long)

# Drug-Protein edges from pre-split data
dp_train_edge_index = edges_to_tensor(drugs_interactions_train, 'drug_internal_id', 'protein_id', drug_to_idx, protein_to_idx)
dp_val_edge_index = edges_to_tensor(drugs_interactions_validation, 'drug_internal_id', 'protein_id', drug_to_idx, protein_to_idx)
dp_test_edge_index = edges_to_tensor(drugs_interactions_test, 'drug_internal_id', 'protein_id', drug_to_idx, protein_to_idx)

print(f"  Drug-Protein Training edges:   {dp_train_edge_index.shape[1]:,}")
print(f"  Drug-Protein Validation edges: {dp_val_edge_index.shape[1]:,}")
print(f"  Drug-Protein Test edges:       {dp_test_edge_index.shape[1]:,}")


STEP 2: PROCESS DRUG-PROTEIN EDGES (TIME-AWARE SPLIT)


  Drug-Protein Training edges:   10,409
  Drug-Protein Validation edges: 1,901
  Drug-Protein Test edges:       1,901


In [17]:
print("\n" + "="*80)
print("STEP 3: PROCESS DRUG-INDICATION EDGES (RANDOM SPLIT)")
print("="*80)

# Create edges from drug_indications
di_edges = edges_to_tensor(drug_indications, 'drug_internal_id', 'effect_id', drug_to_idx, effect_to_idx)
print(f"  Total drug-indication edges: {di_edges.shape[1]:,}")

# Random 80/10/10 split (seed for reproducibility)
np.random.seed(42)
num_di_edges = di_edges.shape[1]
perm = np.random.permutation(num_di_edges)
train_size = int(0.8 * num_di_edges)
val_size = int(0.1 * num_di_edges)

di_train_idx = perm[:train_size]
di_val_idx = perm[train_size:train_size + val_size]
di_test_idx = perm[train_size + val_size:]

di_train_edge_index = di_edges[:, di_train_idx]
di_val_edge_index = di_edges[:, di_val_idx]
di_test_edge_index = di_edges[:, di_test_idx]

print(f"  Drug-Indication Training edges:   {di_train_edge_index.shape[1]:,}")
print(f"  Drug-Indication Validation edges: {di_val_edge_index.shape[1]:,}")
print(f"  Drug-Indication Test edges:       {di_test_edge_index.shape[1]:,}")


STEP 3: PROCESS DRUG-INDICATION EDGES (RANDOM SPLIT)
  Total drug-indication edges: 7,086
  Drug-Indication Training edges:   5,668
  Drug-Indication Validation edges: 708
  Drug-Indication Test edges:       710


In [18]:
print("\n" + "="*80)
print("STEP 4: SETUP DYNAMIC NEGATIVE SAMPLING - DRUG-PROTEIN")
print("="*80)

# Create mapping from ChEMBL drug_id to drug_internal_id
chembl_to_internal = dict(zip(drug_nodes['drug_id'], drug_nodes['drug_internal_id']))

def get_existing_edges_set(edge_index):
    """Convert edge_index tensor to set of (src, tgt) tuples."""
    if edge_index.shape[1] == 0:
        return set()
    return set(map(tuple, edge_index.t().numpy()))

# Get nodes that have at least one positive edge in training
train_drugs_with_pos = set(dp_train_edge_index[0].numpy())
train_proteins_with_pos = set(dp_train_edge_index[1].numpy())

print(f"  Drugs with positive edges in train:    {len(train_drugs_with_pos):,}")
print(f"  Proteins with positive edges in train: {len(train_proteins_with_pos):,}")

# Preprocess verified negatives (convert to indices once)
verified_dp_train = []
for _, row in verified_negatives_protien_interactions_train.iterrows():
    drug_chembl = str(row['drug_id'])
    protein_id = str(row['protein_id'])
    if drug_chembl in chembl_to_internal and protein_id in protein_to_idx:
        drug_internal_id = chembl_to_internal[drug_chembl]
        if drug_internal_id in drug_to_idx:
            drug_idx = drug_to_idx[drug_internal_id]
            protein_idx = protein_to_idx[protein_id]
            if drug_idx in train_drugs_with_pos and protein_idx in train_proteins_with_pos:
                verified_dp_train.append((drug_idx, protein_idx))

verified_dp_test = []
for _, row in verified_negatives_protien_interactions_test.iterrows():
    drug_chembl = str(row['drug_id'])
    protein_id = str(row['protein_id'])
    if drug_chembl in chembl_to_internal and protein_id in protein_to_idx:
        drug_internal_id = chembl_to_internal[drug_chembl]
        if drug_internal_id in drug_to_idx:
            drug_idx = drug_to_idx[drug_internal_id]
            protein_idx = protein_to_idx[protein_id]
            if drug_idx in train_drugs_with_pos and protein_idx in train_proteins_with_pos:
                verified_dp_test.append((drug_idx, protein_idx))

verified_dp_train = list(set(verified_dp_train))
verified_dp_test = list(set(verified_dp_test))

print(f"  Verified DP negatives (train): {len(verified_dp_train):,}")
print(f"  Verified DP negatives (test):  {len(verified_dp_test):,}")

def sample_negatives_dp_dynamic(num_samples, verified_negs, existing_edges, valid_srcs, valid_tgts, ratio=0.5):
    """
    Dynamic drug-protein negative sampling.
    50% verified + 50% random (from valid nodes).
    Called during each training step - ensures fresh negatives each time.
    """
    verified_count = min(len(verified_negs), int(num_samples * ratio))
    random_count = num_samples - verified_count
    
    negatives = []
    
    # Add verified negatives
    if verified_negs:
        for edge_idx in np.random.choice(len(verified_negs), min(verified_count, len(verified_negs)), replace=False):
            negatives.append(verified_negs[edge_idx])
    
    # Add random negatives from valid nodes
    attempts = 0
    max_attempts = random_count * 20
    while len(negatives) < num_samples and attempts < max_attempts:
        drug_idx = np.random.choice(list(valid_srcs))
        protein_idx = np.random.choice(list(valid_tgts))
        pair = (drug_idx, protein_idx)
        if pair not in existing_edges and pair not in negatives:
            negatives.append(pair)
        attempts += 1
    
    # Fill remaining
    while len(negatives) < num_samples:
        drug_idx = np.random.choice(list(valid_srcs))
        protein_idx = np.random.choice(list(valid_tgts))
        pair = (drug_idx, protein_idx)
        if pair not in existing_edges and pair not in negatives:
            negatives.append(pair)
    
    return negatives[:num_samples]

# Pre-compute all positive edges for fast lookup during training
all_pos_dp_edges = torch.cat([dp_train_edge_index, dp_val_edge_index, dp_test_edge_index], dim=1)
existing_dp = get_existing_edges_set(all_pos_dp_edges)

print(f"  Total positive DP edges (all splits): {len(existing_dp):,}")


STEP 4: SETUP DYNAMIC NEGATIVE SAMPLING - DRUG-PROTEIN
  Drugs with positive edges in train:    1,402
  Proteins with positive edges in train: 1,966
  Verified DP negatives (train): 43,867
  Verified DP negatives (test):  40,456
  Total positive DP edges (all splits): 14,211


In [19]:
print("\n" + "="*80)
print("STEP 5: SETUP DYNAMIC NEGATIVE SAMPLING - DRUG-INDICATION")
print("="*80)

# Create mapping from effect_name to effect_id
effect_name_to_id = dict(zip(drug_indications['effect_name'], drug_indications['effect_id']))

# Get nodes that have at least one positive edge in training
train_drugs_di = set(di_train_edge_index[0].numpy())
train_effects_di = set(di_train_edge_index[1].numpy())

print(f"  Drugs with positive edges in train:    {len(train_drugs_di):,}")
print(f"  Effects with positive edges in train:  {len(train_effects_di):,}")

# Preprocess hard negatives
hard_neg_edges = []
for _, row in failed_indication_hard_negatives.iterrows():
    drug_chembl = str(row['drug_id'])
    effect_name = str(row['effect_name'])
    if drug_chembl in chembl_to_internal and effect_name in effect_name_to_id:
        drug_internal_id = chembl_to_internal[drug_chembl]
        effect_id = effect_name_to_id[effect_name]
        if drug_internal_id in drug_to_idx and effect_id in effect_to_idx:
            drug_idx = drug_to_idx[drug_internal_id]
            effect_idx = effect_to_idx[effect_id]
            if drug_idx in train_drugs_di and effect_idx in train_effects_di:
                hard_neg_edges.append((drug_idx, effect_idx))

# Preprocess medium negatives
med_neg_edges = []
for _, row in failed_indication_medium_negatives.iterrows():
    drug_chembl = str(row['drug_id'])
    effect_name = str(row['effect_name'])
    if drug_chembl in chembl_to_internal and effect_name in effect_name_to_id:
        drug_internal_id = chembl_to_internal[drug_chembl]
        effect_id = effect_name_to_id[effect_name]
        if drug_internal_id in drug_to_idx and effect_id in effect_to_idx:
            drug_idx = drug_to_idx[drug_internal_id]
            effect_idx = effect_to_idx[effect_id]
            if drug_idx in train_drugs_di and effect_idx in train_effects_di:
                med_neg_edges.append((drug_idx, effect_idx))

hard_neg_edges = list(set(hard_neg_edges))
med_neg_edges = list(set(med_neg_edges))

print(f"  Hard negatives (Phase 3 fails):        {len(hard_neg_edges):,}")
print(f"  Medium negatives (Phase 2 or less):    {len(med_neg_edges):,}")

def sample_negatives_di_dynamic(num_samples, hard_negs, med_negs, existing_edges, valid_srcs, valid_tgts):
    """
    Dynamic drug-indication negative sampling.
    33% hard + 33% medium + 33% random (from valid nodes).
    Called during each training step - ensures fresh negatives each time.
    """
    hard_count = min(len(hard_negs), num_samples // 3)
    med_count = min(len(med_negs), num_samples // 3)
    random_count = num_samples - hard_count - med_count
    
    negatives = []
    
    # Add hard negatives
    if hard_negs:
        for idx in np.random.choice(len(hard_negs), min(hard_count, len(hard_negs)), replace=False):
            negatives.append(hard_negs[idx])
    
    # Add medium negatives
    if med_negs:
        for idx in np.random.choice(len(med_negs), min(med_count, len(med_negs)), replace=False):
            negatives.append(med_negs[idx])
    
    # Add random negatives from valid nodes
    all_neg_set = set(hard_negs + med_negs)
    attempts = 0
    max_attempts = random_count * 20
    
    while len(negatives) < num_samples and attempts < max_attempts:
        drug_idx = np.random.choice(list(valid_srcs))
        effect_idx = np.random.choice(list(valid_tgts))
        pair = (drug_idx, effect_idx)
        if pair not in existing_edges and pair not in all_neg_set and pair not in negatives:
            negatives.append(pair)
        attempts += 1
    
    # Fill remaining
    while len(negatives) < num_samples:
        drug_idx = np.random.choice(list(valid_srcs))
        effect_idx = np.random.choice(list(valid_tgts))
        pair = (drug_idx, effect_idx)
        if pair not in existing_edges and pair not in negatives:
            negatives.append(pair)
    
    return negatives[:num_samples]

# Pre-compute all positive edges for fast lookup during training
all_pos_di_edges = torch.cat([di_train_edge_index, di_val_edge_index, di_test_edge_index], dim=1)
existing_di = get_existing_edges_set(all_pos_di_edges)

print(f"  Total positive DI edges (all splits): {len(existing_di):,}")


STEP 5: SETUP DYNAMIC NEGATIVE SAMPLING - DRUG-INDICATION
  Drugs with positive edges in train:    2,474
  Effects with positive edges in train:  971
  Hard negatives (Phase 3 fails):        4,911
  Medium negatives (Phase 2 or less):    8,783
  Total positive DI edges (all splits): 7,086


In [20]:
print("\n" + "="*80)
print("STEP 6: BUILD PYTORCH GEOMETRIC HETERODATA OBJECTS")
print("="*80)

# ── Identity features: one-hot vectors per node type ──
drug_features = torch.eye(num_drugs)
protein_features = torch.eye(num_proteins)
effect_features = torch.eye(num_effects)

print(f"  Drug identity features:    {drug_features.shape}")
print(f"  Protein identity features: {protein_features.shape}")
print(f"  Effect identity features:  {effect_features.shape}")

# ── Accumulate edges across splits ──
# Train:  train edges only
# Val:    train + val edges   (evaluate on val, but graph includes train)
# Test:   train + val + test  (evaluate on test, but graph includes train+val)

dp_val_cumulative = torch.cat([dp_train_edge_index, dp_val_edge_index], dim=1)
di_val_cumulative = torch.cat([di_train_edge_index, di_val_edge_index], dim=1)

dp_test_cumulative = torch.cat([dp_train_edge_index, dp_val_edge_index, dp_test_edge_index], dim=1)
di_test_cumulative = torch.cat([di_train_edge_index, di_val_edge_index, di_test_edge_index], dim=1)

# ── Build HeteroData objects ──
def build_hetero_data(dp_edges, di_edges, split_name, dp_eval_edges=None, di_eval_edges=None):
    data = HeteroData()
    
    # Identity node features (shared across splits)
    data['drug'].x = drug_features
    data['protein'].x = protein_features
    data['effect'].x = effect_features
    
    # All edges in the graph for message passing
    data['drug', 'binds_to', 'protein'].edge_index = dp_edges
    data['drug', 'treats', 'effect'].edge_index = di_edges
    
    # Store which edges are being evaluated in this split
    if dp_eval_edges is not None:
        data['drug', 'binds_to', 'protein'].eval_edge_index = dp_eval_edges
        data['drug', 'treats', 'effect'].eval_edge_index = di_eval_edges
    
    # Metadata
    data.split = split_name
    data.num_drug_nodes = num_drugs
    data.num_protein_nodes = num_proteins
    data.num_effect_nodes = num_effects
    
    return data

# Train: only train edges (eval = train edges themselves)
hetero_train = build_hetero_data(
    dp_train_edge_index, di_train_edge_index, 'train'
)

# Val: train+val edges in graph, evaluate on val edges
hetero_val = build_hetero_data(
    dp_val_cumulative, di_val_cumulative, 'val',
    dp_eval_edges=dp_val_edge_index, di_eval_edges=di_val_edge_index
)

# Test: train+val+test edges in graph, evaluate on test edges
hetero_test = build_hetero_data(
    dp_test_cumulative, di_test_cumulative, 'test',
    dp_eval_edges=dp_test_edge_index, di_eval_edges=di_test_edge_index
)

for name, graph in [("Train", hetero_train), ("Val", hetero_val), ("Test", hetero_test)]:
    dp_total = graph['drug', 'binds_to', 'protein'].edge_index.shape[1]
    di_total = graph['drug', 'treats', 'effect'].edge_index.shape[1]
    print(f"\n  {name} graph:")
    print(f"    Nodes: {graph.num_nodes}")
    print(f"    DP edges in graph: {dp_total:,}   DI edges in graph: {di_total:,}")
    if hasattr(graph['drug', 'binds_to', 'protein'], 'eval_edge_index'):
        dp_eval = graph['drug', 'binds_to', 'protein'].eval_edge_index.shape[1]
        di_eval = graph['drug', 'treats', 'effect'].eval_edge_index.shape[1]
        print(f"    DP edges to evaluate: {dp_eval:,}   DI edges to evaluate: {di_eval:,}")

# Verify edge indices are within bounds
for name, graph in [("Train", hetero_train), ("Val", hetero_val), ("Test", hetero_test)]:
    dp_ei = graph['drug', 'binds_to', 'protein'].edge_index
    di_ei = graph['drug', 'treats', 'effect'].edge_index
    assert dp_ei[0].max() < num_drugs, f"{name}: DP drug index out of bounds"
    assert dp_ei[1].max() < num_proteins, f"{name}: DP protein index out of bounds"
    assert di_ei[0].max() < num_drugs, f"{name}: DI drug index out of bounds"
    assert di_ei[1].max() < num_effects, f"{name}: DI effect index out of bounds"
print("\n  ✓ All edge indices within node bounds")

print("\n" + "="*80)
print("✓ GRAPH CONSTRUCTION COMPLETE (Identity Features + Dynamic Negative Sampling)")
print("="*80)


STEP 6: BUILD PYTORCH GEOMETRIC HETERODATA OBJECTS
  Drug identity features:    torch.Size([3071, 3071])
  Protein identity features: torch.Size([1966, 1966])
  Effect identity features:  torch.Size([1065, 1065])

  Train graph:
    Nodes: 6102
    DP edges in graph: 10,409   DI edges in graph: 5,668

  Val graph:
    Nodes: 6102
    DP edges in graph: 12,310   DI edges in graph: 6,376
    DP edges to evaluate: 1,901   DI edges to evaluate: 708

  Test graph:
    Nodes: 6102
    DP edges in graph: 14,211   DI edges in graph: 7,086
    DP edges to evaluate: 1,901   DI edges to evaluate: 710

  ✓ All edge indices within node bounds

✓ GRAPH CONSTRUCTION COMPLETE (Identity Features + Dynamic Negative Sampling)


In [21]:
print("\n" + "="*80)
print("STEP 7: SAVE GRAPHS & SAMPLING FUNCTIONS")
print("="*80)

mappings = {
    'drug_to_idx': drug_to_idx,
    'protein_to_idx': protein_to_idx,
    'effect_to_idx': effect_to_idx
}

graphs = {
    'train': hetero_train,
    'val': hetero_val,
    'test': hetero_test
}

# Save sampling configuration
sampling_config = {
    'verified_dp_train': verified_dp_train,
    'verified_dp_test': verified_dp_test,
    'hard_neg_edges': hard_neg_edges,
    'med_neg_edges': med_neg_edges,
    'existing_dp': existing_dp,
    'existing_di': existing_di,
    'train_drugs_with_pos': train_drugs_with_pos,
    'train_proteins_with_pos': train_proteins_with_pos,
    'train_drugs_di': train_drugs_di,
    'train_effects_di': train_effects_di,
}

torch.save(mappings, 'hetero_node_mappings.pt')
torch.save(graphs, 'hetero_graphs.pt')
torch.save(sampling_config, 'dynamic_sampling_config.pt')

print("✓ Saved hetero_node_mappings.pt")
print("✓ Saved hetero_graphs.pt")
print("✓ Saved dynamic_sampling_config.pt")

print("\n" + "="*80)
print("✅ READY FOR TRAINING WITH DYNAMIC NEGATIVE SAMPLING!")
print("="*80)
print("\nImportant:")
print("  • Negatives are NOT pre-sampled")
print("  • Call sample_negatives_dp_dynamic() during each training batch")
print("  • Call sample_negatives_di_dynamic() during each training batch")
print("  • This ensures model doesn't memorize specific hard negatives")
print("  • All negatives come from nodes with at least 1 positive edge")


STEP 7: SAVE GRAPHS & SAMPLING FUNCTIONS
✓ Saved hetero_node_mappings.pt
✓ Saved hetero_graphs.pt
✓ Saved dynamic_sampling_config.pt

✅ READY FOR TRAINING WITH DYNAMIC NEGATIVE SAMPLING!

Important:
  • Negatives are NOT pre-sampled
  • Call sample_negatives_dp_dynamic() during each training batch
  • Call sample_negatives_di_dynamic() during each training batch
  • This ensures model doesn't memorize specific hard negatives
  • All negatives come from nodes with at least 1 positive edge


In [22]:
print("\n" + "="*80)
print("STEP 7: SAVE MAPPINGS & GRAPH DATA")
print("="*80)

mappings = {
    'drug_to_idx': drug_to_idx,
    'protein_to_idx': protein_to_idx,
    'effect_to_idx': effect_to_idx
}

graphs = {
    'train': hetero_train,
    'val': hetero_val,
    'test': hetero_test
}

sampling_config = {
    'verified_dp_train': verified_dp_train,
    'verified_dp_test': verified_dp_test,
    'existing_dp': existing_dp,
    'train_drugs_with_pos': train_drugs_with_pos,
    'train_proteins_with_pos': train_proteins_with_pos,
    'hard_neg_edges': hard_neg_edges,
    'med_neg_edges': med_neg_edges,
    'existing_di': existing_di,
    'train_drugs_di': train_drugs_di,
    'train_effects_di': train_effects_di,
}

torch.save(mappings, 'hetero_node_mappings.pt')
torch.save(graphs, 'hetero_graphs.pt')
torch.save(sampling_config, 'dynamic_sampling_config.pt')

print("✓ Saved hetero_node_mappings.pt")
print("✓ Saved hetero_graphs.pt")
print("✓ Saved dynamic_sampling_config.pt")

print(f"\nGraph summary:")
print(f"  Nodes: {num_drugs:,} drugs + {num_proteins:,} proteins + {num_effects:,} effects = {total_nodes:,} total")
print(f"  Features: identity (torch.eye)")
print(f"  Edges (DP): train={dp_train_edge_index.shape[1]:,} / val={dp_val_edge_index.shape[1]:,} / test={dp_test_edge_index.shape[1]:,}")
print(f"  Edges (DI): train={di_train_edge_index.shape[1]:,} / val={di_val_edge_index.shape[1]:,} / test={di_test_edge_index.shape[1]:,}")

print("\n" + "="*80)
print("✅ READY FOR MODEL TRAINING!")
print("="*80)
print("\nVariables available:")
print("  • hetero_train, hetero_val, hetero_test (HeteroData objects)")
print("  • drug_to_idx, protein_to_idx, effect_to_idx (mappings)")
print("  • sample_negatives_dp_dynamic(), sample_negatives_di_dynamic() (call per batch)")
print("  • Negatives are NOT pre-stored — sampled fresh each training step")


STEP 7: SAVE MAPPINGS & GRAPH DATA
✓ Saved hetero_node_mappings.pt
✓ Saved hetero_graphs.pt
✓ Saved dynamic_sampling_config.pt

Graph summary:
  Nodes: 3,071 drugs + 1,966 proteins + 1,065 effects = 6,102 total
  Features: identity (torch.eye)
  Edges (DP): train=10,409 / val=1,901 / test=1,901
  Edges (DI): train=5,668 / val=708 / test=710

✅ READY FOR MODEL TRAINING!

Variables available:
  • hetero_train, hetero_val, hetero_test (HeteroData objects)
  • drug_to_idx, protein_to_idx, effect_to_idx (mappings)
  • sample_negatives_dp_dynamic(), sample_negatives_di_dynamic() (call per batch)
  • Negatives are NOT pre-stored — sampled fresh each training step


In [23]:
print("\n" + "="*80)
print("EXAMPLE: HOW TO USE DYNAMIC NEGATIVE SAMPLING IN TRAINING")
print("="*80)

# Example usage during training loop:
print("\nDuring training, sample negatives dynamically like this:\n")

print("# For each batch of drug-protein edges:")
sample_neg_dp = sample_negatives_dp_dynamic(
    num_samples=100,
    verified_negs=verified_dp_train,
    existing_edges=existing_dp,
    valid_srcs=train_drugs_with_pos,
    valid_tgts=train_proteins_with_pos
)
print(f"  Generated {len(sample_neg_dp)} fresh negatives for DP")

print("\n# For each batch of drug-indication edges:")
sample_neg_di = sample_negatives_di_dynamic(
    num_samples=100,
    hard_negs=hard_neg_edges,
    med_negs=med_neg_edges,
    existing_edges=existing_di,
    valid_srcs=train_drugs_di,
    valid_tgts=train_effects_di
)
print(f"  Generated {len(sample_neg_di)} fresh negatives for DI")
print("    - This mix includes 33% hard + 33% medium + 33% random")

print("\n✅ These functions will be called EVERY TRAINING STEP/EPOCH")
print("   ensuring the model never memorizes specific negative samples!")

print("\n" + "="*80)
print("KEY VARIABLES FOR MODEL TRAINING:")
print("="*80)
print(f"✓ hetero_train, hetero_val, hetero_test (HeteroData objects)")
print(f"✓ sample_negatives_dp_dynamic() - Call during DP training")
print(f"✓ sample_negatives_di_dynamic() - Call during DI training")
print(f"✓ verified_dp_train, verified_dp_test - Verified negative lists")
print(f"✓ hard_neg_edges, med_neg_edges - Pre-processed failed indications")
print(f"✓ existing_dp, existing_di - All positive edges (for validation)")
print(f"✓ Valid node sets: train_drugs_with_pos, train_proteins_with_pos, etc.")


EXAMPLE: HOW TO USE DYNAMIC NEGATIVE SAMPLING IN TRAINING

During training, sample negatives dynamically like this:

# For each batch of drug-protein edges:
  Generated 100 fresh negatives for DP

# For each batch of drug-indication edges:
  Generated 100 fresh negatives for DI
    - This mix includes 33% hard + 33% medium + 33% random

✅ These functions will be called EVERY TRAINING STEP/EPOCH
   ensuring the model never memorizes specific negative samples!

KEY VARIABLES FOR MODEL TRAINING:
✓ hetero_train, hetero_val, hetero_test (HeteroData objects)
✓ sample_negatives_dp_dynamic() - Call during DP training
✓ sample_negatives_di_dynamic() - Call during DI training
✓ verified_dp_train, verified_dp_test - Verified negative lists
✓ hard_neg_edges, med_neg_edges - Pre-processed failed indications
✓ existing_dp, existing_di - All positive edges (for validation)
✓ Valid node sets: train_drugs_with_pos, train_proteins_with_pos, etc.


In [24]:
import time
from sklearn.metrics import roc_auc_score, average_precision_score

print("="*80)
print("EVALUATION METRICS (Reviewer-Requested)")
print("="*80)

def print_metrics(metrics, prefix=""):
    """Pretty-print a metrics dict."""
    print(f"  {prefix}ROC-AUC: {metrics['roc_auc']:.4f}   PR-AUC: {metrics['pr_auc']:.4f}")
    print(f"  {prefix}MRR:     {metrics['mrr']:.4f}   Hits@1: {metrics['hits@1']:.4f}  "
          f"Hits@3: {metrics['hits@3']:.4f}  Hits@10: {metrics['hits@10']:.4f}")

def get_peak_vram_mb():
    """Return peak GPU VRAM allocated in MB (0 if CPU-only)."""
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024**2
    return 0.0

def reset_vram_tracker():
    """Reset the peak VRAM counter before each epoch."""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

print("  ✓ Evaluation utilities ready")
print("  Protocol: Standard filtered ranking (Bordes et al., 2013)")
print("    - Each positive edge ranked against ALL entities of the target type")
print("    - Other known positives filtered out (score → -inf)")
print("    - MRR, Hits@k computed per-edge then averaged")
print("    - ROC-AUC, PR-AUC derived from the full ranking scores")

EVALUATION METRICS (Reviewer-Requested)
  ✓ Evaluation utilities ready
  Protocol: Standard filtered ranking (Bordes et al., 2013)
    - Each positive edge ranked against ALL entities of the target type
    - Other known positives filtered out (score → -inf)
    - MRR, Hits@k computed per-edge then averaged
    - ROC-AUC, PR-AUC derived from the full ranking scores


In [25]:
import torch.nn as nn
import torch.nn.functional as F

print("="*80)
print("DISTMULT MODEL FOR HETEROGENEOUS KNOWLEDGE GRAPH")
print("="*80)

class DistMult(nn.Module):
    """
    DistMult for a heterogeneous graph with 3 node types and 2 relation types.
    
    DistMult uses diagonal relation matrices. For triple (h, r, t):
    Score = h · diag(r) · t  (higher = more likely true, computed as element-wise product sum)
    
    This is simpler than TransR but often effective. Uses bilinear scoring.
    Reference: Yang et al., "Embedding Entities and Relations for Learning and
    Inference in Knowledge Bases" (ICLR 2015).
    """
    def __init__(self, num_drugs, num_proteins, num_effects,
                 embed_dim=128, margin=1.0):
        super().__init__()
        self.margin = margin
        self.embed_dim = embed_dim
        self.rel_dim = embed_dim  # For DistMult, relation space = entity space
        
        # Node embeddings
        self.drug_emb    = nn.Embedding(num_drugs,    embed_dim)
        self.protein_emb = nn.Embedding(num_proteins, embed_dim)
        self.effect_emb  = nn.Embedding(num_effects,  embed_dim)
        
        # Relation embeddings (diagonal matrices represented as vectors)
        self.rel_binds_to = nn.Embedding(1, embed_dim)  # drug → protein
        self.rel_treats   = nn.Embedding(1, embed_dim)  # drug → effect
        
        self._init_weights()
    
    def _init_weights(self):
        nn.init.xavier_uniform_(self.drug_emb.weight)
        nn.init.xavier_uniform_(self.protein_emb.weight)
        nn.init.xavier_uniform_(self.effect_emb.weight)
        nn.init.xavier_uniform_(self.rel_binds_to.weight)
        nn.init.xavier_uniform_(self.rel_treats.weight)
    
    def score_dp(self, drug_idx, protein_idx):
        """Score drug-protein pairs using DistMult. Returns score (higher=better)."""
        h = self.drug_emb(drug_idx)           # (batch, embed_dim)
        t = self.protein_emb(protein_idx)     # (batch, embed_dim)
        r = self.rel_binds_to.weight[0]       # (embed_dim,)
        # DistMult score: sum(h_i * r_i * t_i)
        return (h * r * t).sum(dim=-1)
    
    def score_di(self, drug_idx, effect_idx):
        """Score drug-indication pairs using DistMult. Returns score (higher=better)."""
        h = self.drug_emb(drug_idx)           # (batch, embed_dim)
        t = self.effect_emb(effect_idx)       # (batch, embed_dim)
        r = self.rel_treats.weight[0]         # (embed_dim,)
        # DistMult score: sum(h_i * r_i * t_i)
        return (h * r * t).sum(dim=-1)
    
    def margin_loss(self, pos_score, neg_score):
        """Margin ranking loss: max(0, margin - pos_score + neg_score)."""
        return F.relu(self.margin - pos_score + neg_score).mean()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  Device: {device}")

model = DistMult(
    num_drugs=num_drugs,
    num_proteins=num_proteins,
    num_effects=num_effects,
    embed_dim=128,
    margin=1.0
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"  Entity embed dim:  {model.embed_dim}")
print(f"  Total params:      {total_params:,}")
print(f"    Drug emb:        {num_drugs} × {model.embed_dim} = {num_drugs*model.embed_dim:,}")
print(f"    Protein emb:     {num_proteins} × {model.embed_dim} = {num_proteins*model.embed_dim:,}")
print(f"    Effect emb:      {num_effects} × {model.embed_dim} = {num_effects*model.embed_dim:,}")
print(f"    Relation emb:    2 × {model.embed_dim} = {2*model.embed_dim}")
print(f"\n✓ DistMult model ready")

DISTMULT MODEL FOR HETEROGENEOUS KNOWLEDGE GRAPH
  Device: cuda
  Entity embed dim:  128
  Total params:      781,312
    Drug emb:        3071 × 128 = 393,088
    Protein emb:     1966 × 128 = 251,648
    Effect emb:      1065 × 128 = 136,320
    Relation emb:    2 × 128 = 256

✓ DistMult model ready


In [26]:
print("="*80)
print("TRAINING LOOP")
print("="*80)

# ── Hyperparameters ──
NUM_EPOCHS   = 2000
LR           = 1e-3
WEIGHT_DECAY = 1e-5
NEG_RATIO    = 1        # 1 negative per positive
VAL_EVERY    = 10       # validate every N epochs
PATIENCE     = 20       # early stopping (in val checks)
NEG_PER_POS_EVAL = 50   # negatives per positive for ROC-AUC / PR-AUC

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Move training positive edges to device
dp_train_pos = dp_train_edge_index.to(device)   # (2, num_dp_train)
di_train_pos = di_train_edge_index.to(device)   # (2, num_di_train)

n_dp_train = dp_train_pos.shape[1]
n_di_train = di_train_pos.shape[1]

print(f"  Epochs: {NUM_EPOCHS},  LR: {LR},  Neg ratio: {NEG_RATIO}")
print(f"  DP train edges: {n_dp_train:,},  DI train edges: {n_di_train:,}")
print(f"  Validate every {VAL_EVERY} epochs,  Patience: {PATIENCE} checks")
print(f"  Neg per pos (AUC eval): {NEG_PER_POS_EVAL}")

# ── Helper: build negative tensor for one training epoch ──
def epoch_negatives_dp():
    """Sample dynamic negatives for drug-protein (one per positive)."""
    negs = sample_negatives_dp_dynamic(
        num_samples=n_dp_train * NEG_RATIO,
        verified_negs=verified_dp_train,
        existing_edges=existing_dp,
        valid_srcs=train_drugs_with_pos,
        valid_tgts=train_proteins_with_pos
    )
    neg_t = torch.tensor(negs, dtype=torch.long).t().to(device)  # (2, N)
    return neg_t

def epoch_negatives_di():
    """Sample dynamic negatives for drug-indication (one per positive)."""
    negs = sample_negatives_di_dynamic(
        num_samples=n_di_train * NEG_RATIO,
        hard_negs=hard_neg_edges,
        med_negs=med_neg_edges,
        existing_edges=existing_di,
        valid_srcs=train_drugs_di,
        valid_tgts=train_effects_di
    )
    neg_t = torch.tensor(negs, dtype=torch.long).t().to(device)  # (2, N)
    return neg_t

# ── Standard filtered ranking evaluation (Bordes et al., 2013) ──
@torch.no_grad()
def evaluate_filtered(pos_edge_index, relation_type, all_positive_set,
                      neg_per_pos=NEG_PER_POS_EVAL):
    """
    Standard filtered ranking evaluation for link prediction.
    
    Ranking metrics (MRR, Hits@k):
      For each positive (h, t), score ALL possible targets t', filter known
      positives, rank = 1 + count(filtered > true).  Standard KG protocol.
    
    Classification metrics (ROC-AUC, PR-AUC):
      For each positive (h, t), sample `neg_per_pos` random negatives from
      the full entity pool (excluding known positives for h).  This gives a
      balanced positive rate of 1/(1+neg_per_pos) ≈ 2% and interpretable AUC
      values.  Standard in KGAT, CompGCN, etc.
    
    Args:
        pos_edge_index:   (2, N) tensor of positive edges to evaluate
        relation_type:    'dp' (drug→protein) or 'di' (drug→effect)
        all_positive_set: set of ALL known positive (src, tgt) tuples for filtering
        neg_per_pos:      number of sampled negatives per positive for AUC metrics
    
    Returns:
        dict with ROC-AUC, PR-AUC, MRR, Hits@1, Hits@3, Hits@10
    """
    model.eval()
    
    if relation_type == 'dp':
        n_tails = num_proteins
        score_fn = model.score_dp
    else:
        n_tails = num_effects
        score_fn = model.score_di
    
    # Precompute head → set of known positive tails (for fast filtering)
    head_to_pos_tails = {}
    for (h, t) in all_positive_set:
        if h not in head_to_pos_tails:
            head_to_pos_tails[h] = set()
        head_to_pos_tails[h].add(t)
    
    n_pos = pos_edge_index.shape[1]
    all_tails = torch.arange(n_tails, device=device)
    
    reciprocal_ranks = []
    hits_at = {1: 0, 3: 0, 10: 0}
    
    # For classification metrics: collect per-edge pos/neg scores
    auc_y_true = []
    auc_y_score = []
    
    EVAL_BATCH = 256
    for start in range(0, n_pos, EVAL_BATCH):
        end = min(start + EVAL_BATCH, n_pos)
        batch_heads = pos_edge_index[0, start:end]         # (B,)
        batch_true_tails = pos_edge_index[1, start:end]    # (B,)
        B = end - start
        
        # Score all (h_i, t_j) pairs → shape (B, n_tails)
        heads_exp = batch_heads.unsqueeze(1).expand(B, n_tails).reshape(-1)
        tails_exp = all_tails.unsqueeze(0).expand(B, n_tails).reshape(-1)
        scores = score_fn(heads_exp, tails_exp).reshape(B, n_tails).cpu().numpy()
        
        for i in range(B):
            h = batch_heads[i].item()
            true_t = batch_true_tails[i].item()
            true_score = scores[i, true_t]
            known_tails = head_to_pos_tails.get(h, set())
            
            # ── Ranking metrics: filter known positives ──
            filtered_scores = scores[i].copy()
            for kt in known_tails:
                if kt != true_t and kt < n_tails:
                    filtered_scores[kt] = -np.inf
            
            rank = 1 + int((filtered_scores > true_score).sum())
            reciprocal_ranks.append(1.0 / rank)
            for k in hits_at:
                if rank <= k:
                    hits_at[k] += 1
            
            # ── Classification metrics: sample neg_per_pos negatives ──
            # Build pool of valid negative tails (exclude known positives)
            neg_candidates = []
            for t_idx in range(n_tails):
                if t_idx != true_t and t_idx not in known_tails:
                    neg_candidates.append(t_idx)
            
            # Sample neg_per_pos from the pool
            k = min(neg_per_pos, len(neg_candidates))
            sampled_neg_indices = np.random.choice(neg_candidates, size=k, replace=False)
            sampled_neg_scores = scores[i, sampled_neg_indices]
            
            # 1 positive + k negatives for this edge
            auc_y_true.append(1.0)
            auc_y_score.append(true_score)
            for ns in sampled_neg_scores:
                auc_y_true.append(0.0)
                auc_y_score.append(ns)
    
    metrics = {}
    metrics['mrr']     = float(np.mean(reciprocal_ranks))
    metrics['hits@1']  = hits_at[1] / n_pos
    metrics['hits@3']  = hits_at[3] / n_pos
    metrics['hits@10'] = hits_at[10] / n_pos
    
    # Classification metrics from balanced per-edge sampling
    auc_y_true = np.array(auc_y_true)
    auc_y_score = np.array(auc_y_score)
    metrics['roc_auc'] = roc_auc_score(auc_y_true, auc_y_score)
    metrics['pr_auc']  = average_precision_score(auc_y_true, auc_y_score)
    
    return metrics

# ── Training ──
best_val_mrr = 0.0
patience_counter = 0
history = []

total_train_time = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    reset_vram_tracker()
    t0 = time.time()
    
    model.train()
    
    # 1. Sample fresh negatives
    dp_neg = epoch_negatives_dp()
    di_neg = epoch_negatives_di()
    
    # 2. Forward pass — drug-protein (DistMult scoring)
    dp_pos_score = model.score_dp(dp_train_pos[0], dp_train_pos[1])  # (num_pos,)
    dp_neg_score = model.score_dp(dp_neg[0], dp_neg[1])              # (num_neg,)
    loss_dp = model.margin_loss(dp_pos_score, dp_neg_score)
    
    # 3. Forward pass — drug-indication (DistMult scoring)
    di_pos_score = model.score_di(di_train_pos[0], di_train_pos[1])  # (num_pos,)
    di_neg_score = model.score_di(di_neg[0], di_neg[1])              # (num_neg,)
    loss_di = model.margin_loss(di_pos_score, di_neg_score)
    
    # 4. Combined loss + backward
    loss = loss_dp + loss_di
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    epoch_time = time.time() - t0
    total_train_time += epoch_time
    peak_vram = get_peak_vram_mb()
    
    # ── Validation (standard filtered ranking — deterministic, no sampling noise) ──
    if epoch % VAL_EVERY == 0 or epoch == 1:
        val_dp_pos = dp_val_edge_index.to(device)
        val_di_pos = di_val_edge_index.to(device)
        val_metrics = {
            'dp': evaluate_filtered(val_dp_pos, 'dp', existing_dp),
            'di': evaluate_filtered(val_di_pos, 'di', existing_di)
        }
        
        avg_mrr = (val_metrics['dp']['mrr'] + val_metrics['di']['mrr']) / 2
        
        history.append({
            'epoch': epoch,
            'loss': loss.item(),
            'loss_dp': loss_dp.item(),
            'loss_di': loss_di.item(),
            'epoch_time': epoch_time,
            'peak_vram_mb': peak_vram,
            'val_dp': val_metrics['dp'],
            'val_di': val_metrics['di'],
        })
        
        print(f"Epoch {epoch:>3d}/{NUM_EPOCHS}  "
              f"Loss: {loss.item():.4f} (DP={loss_dp.item():.4f} DI={loss_di.item():.4f})  "
              f"Time: {epoch_time:.2f}s  VRAM: {peak_vram:.0f}MB")
        print(f"  Val DP ─ ROC-AUC: {val_metrics['dp']['roc_auc']:.4f}  "
              f"PR-AUC: {val_metrics['dp']['pr_auc']:.4f}  "
              f"MRR: {val_metrics['dp']['mrr']:.4f}  "
              f"H@10: {val_metrics['dp']['hits@10']:.4f}")
        print(f"  Val DI ─ ROC-AUC: {val_metrics['di']['roc_auc']:.4f}  "
              f"PR-AUC: {val_metrics['di']['pr_auc']:.4f}  "
              f"MRR: {val_metrics['di']['mrr']:.4f}  "
              f"H@10: {val_metrics['di']['hits@10']:.4f}")
        
        # Early stopping on average MRR
        if avg_mrr > best_val_mrr:
            best_val_mrr = avg_mrr
            patience_counter = 0
            torch.save(model.state_dict(), 'distmult_model_checkpoint.pt')
            print(f"  ★ New best avg MRR: {avg_mrr:.4f} — model saved")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"\n  Early stopping at epoch {epoch} (no improvement for {PATIENCE} checks)")
                break

print(f"\n{'='*80}")
print(f"TRAINING COMPLETE")
print(f"{'='*80}")
print(f"  Total training time:  {total_train_time:.1f}s  ({total_train_time/60:.1f} min)")
print(f"  Best val avg MRR:     {best_val_mrr:.4f}")
print(f"  Final peak VRAM:      {get_peak_vram_mb():.0f} MB")

TRAINING LOOP
  Epochs: 2000,  LR: 0.001,  Neg ratio: 1
  DP train edges: 10,409,  DI train edges: 5,668
  Validate every 10 epochs,  Patience: 20 checks
  Neg per pos (AUC eval): 50
Epoch   1/2000  Loss: 2.0000 (DP=1.0000 DI=1.0000)  Time: 1.70s  VRAM: 63MB
  Val DP ─ ROC-AUC: 0.5067  PR-AUC: 0.0201  MRR: 0.0049  H@10: 0.0068
  Val DI ─ ROC-AUC: 0.5011  PR-AUC: 0.0196  MRR: 0.0071  H@10: 0.0085
  ★ New best avg MRR: 0.0060 — model saved
Epoch  10/2000  Loss: 1.9958 (DP=0.9988 DI=0.9970)  Time: 1.26s  VRAM: 69MB
  Val DP ─ ROC-AUC: 0.5184  PR-AUC: 0.0229  MRR: 0.0080  H@10: 0.0110
  Val DI ─ ROC-AUC: 0.5165  PR-AUC: 0.0216  MRR: 0.0077  H@10: 0.0099
  ★ New best avg MRR: 0.0078 — model saved
Epoch  20/2000  Loss: 1.9892 (DP=0.9967 DI=0.9925)  Time: 1.32s  VRAM: 69MB
  Val DP ─ ROC-AUC: 0.5493  PR-AUC: 0.0322  MRR: 0.0167  H@10: 0.0300
  Val DI ─ ROC-AUC: 0.5525  PR-AUC: 0.0277  MRR: 0.0133  H@10: 0.0226
  ★ New best avg MRR: 0.0150 — model saved
Epoch  30/2000  Loss: 1.9782 (DP=0.9927 

In [27]:
print("="*80)
print("FINAL TEST EVALUATION (Standard Filtered Ranking)")
print("="*80)

# Load best checkpoint
model.load_state_dict(torch.load('distmult_model_checkpoint.pt', map_location=device, weights_only=True))
model.eval()
print("  ✓ Loaded best DistMult model checkpoint\n")

# Evaluate on test edges using standard filtered ranking
# Each positive is ranked against ALL entities of the target type,
# with other known positives filtered out (score → -inf).
test_dp_pos = dp_test_edge_index.to(device)
test_di_pos = di_test_edge_index.to(device)

print("  Evaluating DP (drug→protein) ...")
test_dp_metrics = evaluate_filtered(test_dp_pos, 'dp', existing_dp)
print("  Evaluating DI (drug→effect) ...")
test_di_metrics = evaluate_filtered(test_di_pos, 'di', existing_di)
test_metrics = {'dp': test_dp_metrics, 'di': test_di_metrics}

print("\n  Drug-Protein (binds_to):")
print_metrics(test_metrics['dp'], prefix="    ")
print()
print("  Drug-Indication (treats):")
print_metrics(test_metrics['di'], prefix="    ")

# ── Summary table for the paper ──
print(f"\n{'='*80}")
print("RESULTS TABLE (for paper)")
print(f"{'='*80}")
print(f"{'Metric':<12} {'DP (binds_to)':>14} {'DI (treats)':>14}")
print(f"{'-'*40}")
for m in ['roc_auc', 'pr_auc', 'mrr', 'hits@1', 'hits@3', 'hits@10']:
    dp_val = test_metrics['dp'][m]
    di_val = test_metrics['di'][m]
    print(f"{m:<12} {dp_val:>14.4f} {di_val:>14.4f}")

print(f"\n{'='*80}")
print("COMPUTATIONAL COST (for paper)")
print(f"{'='*80}")
print(f"  Total training time:   {total_train_time:.1f}s ({total_train_time/60:.1f} min)")
print(f"  Avg epoch time:        {total_train_time/len(history)/VAL_EVERY:.2f}s") if history else None
print(f"  Peak GPU VRAM:         {get_peak_vram_mb():.0f} MB")
print(f"  Device:                {device}")
print(f"  Model parameters:      {sum(p.numel() for p in model.parameters()):,}")
print(f"  Entity embed dim:      {model.embed_dim}")
print(f"  Relation embeddings:   2 × {model.embed_dim}")
print(f"  Graph: {num_drugs} drugs + {num_proteins} proteins + {num_effects} effects = {total_nodes} nodes")
print(f"\n  Evaluation protocol:   Standard filtered ranking (Bordes et al., 2013)")
print(f"  DP ranking pool:       {num_proteins} proteins  |  DI ranking pool: {num_effects} effects")

FINAL TEST EVALUATION (Standard Filtered Ranking)
  ✓ Loaded best DistMult model checkpoint

  Evaluating DP (drug→protein) ...
  Evaluating DI (drug→effect) ...

  Drug-Protein (binds_to):
      ROC-AUC: 0.7532   PR-AUC: 0.2314
      MRR:     0.1142   Hits@1: 0.0568  Hits@3: 0.1241  Hits@10: 0.2241

  Drug-Indication (treats):
      ROC-AUC: 0.7717   PR-AUC: 0.3246
      MRR:     0.2394   Hits@1: 0.1676  Hits@3: 0.2606  Hits@10: 0.3803

RESULTS TABLE (for paper)
Metric        DP (binds_to)    DI (treats)
----------------------------------------
roc_auc              0.7532         0.7717
pr_auc               0.2314         0.3246
mrr                  0.1142         0.2394
hits@1               0.0568         0.1676
hits@3               0.1241         0.2606
hits@10              0.2241         0.3803

COMPUTATIONAL COST (for paper)
  Total training time:   1313.1s (21.9 min)
  Avg epoch time:        1.24s
  Peak GPU VRAM:         1005 MB
  Device:                cuda
  Model parameters: 